# 06 Rain-to-price event study: not computable

Question (T142 only): do heavy-rain events move vegetable prices, and does that survive
pre-trend, placebo and clustered-SE checks? **It cannot be answered yet.** The price sources are
blocked: S22 exposes no price row and S23 is a one-day snapshot with an unverified unit (D120,
D126). There are zero price rows, so the event study has n = 0: every estimate is reported as
"not computable", never 0, and no chart is drawn from invented data.

The method is ready in `atlas.analysis.event_study` (tested on hand-checkable inputs): unit and
period fixed effects with event-time dummies (reference -1), CR1 cluster-robust covariance with
t(G-1) 95% CIs, a pre-trend check (every lead's CI covers 0) and a placebo check (the same design
on event dates moved to non-event years must give a CI that covers 0). If either check fails, or
there is no data, the result is labelled **association only** (D127). Every event-study chart must
show the 95% CI and its source and vintage (`coef_chart` refuses otherwise).

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/06_rain_price/` (or `ATLAS_EDA_OUT`).

In [ ]:
import json
import os
from pathlib import Path

from atlas.analysis.event_study import verdict
from atlas.config import load_state

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/06_rain_price"))
OUT.mkdir(parents=True, exist_ok=True)
ka = load_state(Path("configs/states/ka.yaml"))
prices = {s: ka.source(s) for s in ("S22", "S23")}
price_rows = 0  # both sources disabled: their adapters raise SourceUnavailable (D120)
assert not any(c.enabled for c in prices.values())
summary = {
    "price_sources": {
        s: {"enabled": c.enabled, "verify_task": c.verify_task} for s, c in prices.items()
    },
    "n_price_rows": price_rows,
    "n_markets": 0,
    "n_rain_events": 0,
    "estimate": "not computable",
    "pretrend": "not computable",
    "placebo": "not computable",
    "clustered_se": "not computable",
    "identification": verdict([], None),
    "charts": [],
    "method": {
        "design": "unit and period fixed effects, event-time dummies, reference -1",
        "se": "CR1 cluster-robust by market, 95% CI with t(G-1)",
        "pretrend": "every lead's 95% CI covers 0",
        "placebo": "same design on event dates moved to non-event years; CI covers 0",
        "label_rule": "association only unless both checks pass",
    },
    "retry": "when S22 or S23 gives weekly modal prices for 30 markets with a stated unit (D126)",
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))